In [6]:
import sys, pathlib 
ROOT = pathlib.Path.cwd().parent 
sys.path.insert(0, str(ROOT))
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import brier_score_loss

from src.detection.detectors import load_all

D = "../data/processed/cicids2017"
X = joblib.load(f"{D}/X_test_processed.joblib")
y = np.asarray(joblib.load(f"{D}/y_test.joblib"))
dets = load_all("../artifacts/cicids2017")
print(X.shape, y.shape, y.mean())

idx_A, idx_B = train_test_split(np.arange(len(y)), test_size=0.5, stratify=y, random_state=42)
y_A, y_B = y[idx_A], y[idx_B]
print(len(idx_A), len(idx_B), y_A.mean(), y_B.mean())



(565580, 70) (565580,) 0.19680858587644542
282790 282790 0.19681035397291277 0.19680681777997808


In [16]:
def reliability(y_true, p, n_bins=10):
    idx = np.digitize(p, np.linspace(0, 1, n_bins + 1)[1:-1])
    rows = []
    for b in range(n_bins):
        m = idx == b
        if m.sum():
            rows.append((b, p[m].mean(), y_true[m].mean(), int(m.sum())))
    return pd.DataFrame(rows, columns=["bin", "predicted", "actual", "n"])


cal, p_A, p_B = {}, {}, {}
for name in ("isolation_forest", "mahalanobis", "autoencoder"):
    s = np.asarray(dets[name].predict(X)["score"], dtype=float)
    s = np.nan_to_num(s, nan=0.0, posinf=1e12)
    cal[name] = IsotonicRegression(out_of_bounds="clip", y_min=0.01, y_max=0.99).fit(s[idx_A], y_A)
    p_A[name], p_B[name] = cal[name].predict(s[idx_A]), cal[name].predict(s[idx_B])
    print(name, "Brier half B:", round(brier_score_loss(y_B, p_B[name]), 4))
    print(reliability(y_B, p_B[name]).round(3).to_string(index=False))

isolation_forest Brier half B: 0.1192
 bin  predicted  actual      n
   0      0.025   0.022  91049
   1      0.193   0.192 148599
   2      0.214   0.210    319
   3      0.394   0.390  21903
   4      0.456   0.479     48
   5      0.531   0.589    175
   6      0.663   0.667    523
   7      0.794   0.794  20174


/Users/sayoowo5/CS4360-Senior-Experience-Project/src/detection/detectors.py:62: RuntimeWarning: divide by zero encountered in matmul
  return ((d @ self.cov_inv) * d).sum(axis=1)
/Users/sayoowo5/CS4360-Senior-Experience-Project/src/detection/detectors.py:62: RuntimeWarning: overflow encountered in matmul
  return ((d @ self.cov_inv) * d).sum(axis=1)
/Users/sayoowo5/CS4360-Senior-Experience-Project/src/detection/detectors.py:62: RuntimeWarning: invalid value encountered in matmul
  return ((d @ self.cov_inv) * d).sum(axis=1)


mahalanobis Brier half B: 0.104
 bin  predicted  actual      n
   0      0.014   0.006 116205
   2      0.226   0.225 130518
   3      0.387   0.393   1538
   4      0.421   0.430   7325
   5      0.538   0.518   2468
   6      0.660   0.700    340
   7      0.751   0.750   6747
   8      0.802   0.805   6499
   9      0.903   0.908  11150
autoencoder Brier half B: 0.0977
 bin  predicted  actual     n
   0      0.010   0.000 87245
   1      0.164   0.164 94416
   2      0.219   0.218 73176
   3      0.332   0.334  1897
   6      0.601   0.624  3045
   7      0.743   0.734  1661
   8      0.853   0.872  1721
   9      0.966   0.967 19629


In [18]:
import os

p_avg_A = (p_A["mahalanobis"] + p_A["autoencoder"]) / 2
p_avg_B = (p_B["mahalanobis"] + p_B["autoencoder"])/ 2
comb = IsotonicRegression(out_of_bounds="clip", y_min=0.01,y_max=0.99).fit(p_avg_A,y_A)
pc_A, pc_B = comb.predict(p_avg_A), comb.predict(p_avg_B)
print("combined prior Brier, half B:", round(brier_score_loss(y_B, pc_B), 4))
print(reliability(y_B, pc_B).round(3).to_string(index=False))

mask_A =(pc_A > 0.2) & (pc_A < 0.9)
mask_B = (pc_B > 0.2) & (pc_B <0.9)

mask_A = (pc_A > 0.2) & (pc_A < 0.9)
mask_B = (pc_B > 0.2) & (pc_B < 0.9)
print("band sizes A/B:", mask_A.sum(), mask_B.sum())
print("attack rate in band A/B:", y_A[mask_A].mean().round(3), y_B[mask_B].mean().round(3))
print("share of all attacks inside the band (half B):", (y_B[mask_B].sum() / y_B.sum()).round(3))
print("attacks the prior calls confident-normal (<=0.2):", int(y_B[pc_B <= 0.2].sum()), "of", int(y_B.sum()))

labels_B = np.asarray(joblib.load(f"{D}/attack_labels_test.joblib")).astype(str)[idx_B]
tab = pd.DataFrame({"label": labels_B, "below_band": pc_B <= 0.2, "in_band": mask_B, "above_band": pc_B >= 0.9})
tab = tab[tab.label != "BENIGN"]
out = tab.groupby("label")[["below_band", "in_band", "above_band"]].mean().round(3)
out["rows"] = tab.groupby("label").size()
print(out.sort_values("rows", ascending=False).to_string())

A_DIR = "../artifacts/cicids2017"
for name in cal:
    joblib.dump(cal[name], f"{A_DIR}/calib_{name}.joblib")
joblib.dump(comb, f"{A_DIR}/agent_calibrator.joblib")
print(sorted(os.listdir(A_DIR)))

combined prior Brier, half B: 0.0944
 bin  predicted  actual      n
   0      0.010   0.002 119970
   1      0.188   0.185  16845
   2      0.242   0.241 119287
   5      0.531   0.543   2980
   6      0.616   0.627   2108
   8      0.803   0.793   1443
   9      0.963   0.967  20157
band sizes A/B: 125997 125806
attack rate in band A/B: 0.261 0.261
share of all attacks inside the band (half B): 0.59
attacks the prior calls confident-normal (<=0.2): 3307 of 55655
                            below_band  in_band  above_band   rows
label                                                             
DoS Hulk                         0.083    0.330       0.587  23051
PortScan                         0.001    0.999       0.000  15865
DDoS                             0.032    0.551       0.417  12733
DoS GoldenEye                    0.000    0.750       0.250   1046
FTP-Patator                      0.001    0.999       0.000    800
SSH-Patator                      0.982    0.018       0.000    

In [19]:
import gc
from src.data.cicids_loader import get_cicids_files, load_cicids_file
from src.data.prepare_cicids_data import split_file_data
raw_dir = ROOT/ "data" /"raw" / "cicids2017"/ "MachineLearningCVE"
parts = []
for f in get_cicids_files(raw_dir):
    data = load_cicids_file(f)
    _, test_part = split_file_data(data)
    parts.append(test_part)
    del data
    gc.collect()
raw_test = pd.concat(parts, ignore_index=True)
del parts

feature_cols = [c for c in raw_test.columns if c != "Label"]
pre = joblib.load(f"{D}/preprocessor.joblib")
check = pre.transform(raw_test[feature_cols]).astype(np.float32)
print(raw_test.shape, "rows line up with X_test:", np.array_equal(check, X))
raw_test = raw_test[feature_cols]
del check

(565580, 79) rows line up with X_test: True


In [20]:
from sklearn.tree import DecisionTreeClassifier, export_text

rawA = raw_test.iloc[idx_A][mask_A]
rawB = raw_test.iloc[idx_B][mask_B]
yA_zone, yB_zone = y_A[mask_A], y_B[mask_B]

tree = DecisionTreeClassifier(max_depth=2, min_samples_leaf=100, random_state=42).fit(rawA, yA_zone)
print(export_text(tree, feature_names=list(rawA.columns), show_weights=True))

def leaf_table(leaf, y_zone):
    t = pd.crosstab(leaf, y_zone)
    t.columns = ["normal", "attack"]
    t["attack_rate"] = t["attack"] / (t["attack"] + t["normal"])
    return t

leafA, leafB = tree.apply(rawA), tree.apply(rawB)

tabA = leaf_table(leafA, yA_zone)
k = len(tabA)
tabA["LR"] = ((tabA["attack"] + 1) / (tabA["attack"].sum() + k)) / \
             ((tabA["normal"] + 1) / (tabA["normal"].sum() + k))
print("HALF A (where the ratios are estimated)")
print(tabA.round(4))
print("HALF B (check only, no re-tuning)")
print(leaf_table(leafB, yB_zone).round(3))

|--- Total Length of Fwd Packets <= 2.50
|   |--- Bwd Header Length <= 22.00
|   |   |--- weights: [4051.00, 20895.00] class: 1
|   |--- Bwd Header Length >  22.00
|   |   |--- weights: [7716.00, 315.00] class: 0
|--- Total Length of Fwd Packets >  2.50
|   |--- Bwd Packet Length Std <= 1495.29
|   |   |--- weights: [81008.00, 6370.00] class: 0
|   |--- Bwd Packet Length Std >  1495.29
|   |   |--- weights: [294.00, 5348.00] class: 1

HALF A (where the ratios are estimated)
       normal  attack  attack_rate       LR
row_0                                      
2        4051   20895       0.8376  14.5747
3        7716     315       0.0392   0.1157
5       81008    6370       0.0729   0.2223
6         294    5348       0.9479  51.2456
HALF B (check only, no re-tuning)
       normal  attack  attack_rate
row_0                             
2        4127   20721        0.834
3        7908     355        0.043
5       80652    6356        0.073
6         255    5432        0.955


In [21]:
ifA = p_A["isolation_forest"][mask_A] > 0.5
ifB = p_B["isolation_forest"][mask_B] > 0.5

def if_table(leaf, flag, y_zone):
    rows = []
    for g in sorted(set(leaf)):
        in_g = leaf == g
        n_att = (y_zone[in_g] == 1).sum()
        n_nor = (y_zone[in_g] == 0).sum()
        for f in (False, True):
            s = in_g & (flag == f)
            a = (y_zone[s] == 1).sum()
            n0 = (y_zone[s] == 0).sum()
            lr = ((a + 1) / (n_att + 2)) / ((n0 + 1) / (n_nor + 2))
            rows.append((g, f, n0, a, round(a / max(a + n0, 1), 3), round(lr, 3)))
    return pd.DataFrame(rows, columns=["leaf", "if_says_attack", "normal", "attack", "attack_rate", "LR"])

tIF_A, tIF_B = if_table(leafA, ifA, yA_zone), if_table(leafB, ifB, yB_zone)
print("HALF A"); print(tIF_A.to_string(index=False))
print("HALF B"); print(tIF_B.to_string(index=False))

# the rule, decided on half A only
use_if = []
for g in sorted(set(leafA)):
    no  = tIF_A[(tIF_A.leaf == g) & (~tIF_A.if_says_attack)].iloc[0]
    yes = tIF_A[(tIF_A.leaf == g) & (tIF_A.if_says_attack)].iloc[0]
    ok = (no.normal + no.attack >= 100) and (yes.normal + yes.attack >= 100) \
         and (yes.attack_rate - no.attack_rate >= 0.10)
    print("group", g, "use Isolation Forest evidence:", ok)
    if ok:
        use_if.append(g)

HALF A
 leaf  if_says_attack  normal  attack  attack_rate     LR
    2           False    4051   20895        0.838  1.000
    2            True       0       0        0.000  0.194
    3           False    7715     309        0.039  0.978
    3            True       1       6        0.857 85.215
    5           False   76907    6256        0.075  1.034
    5            True    4101     114        0.027  0.356
    6           False     280    4281        0.939  0.843
    6            True      14    1067        0.987  3.939
HALF B
 leaf  if_says_attack  normal  attack  attack_rate      LR
    2           False    4127   20721        0.834   1.000
    2            True       0       0        0.000   0.199
    3           False    7908     351        0.042   0.986
    3            True       0       4        1.000 110.784
    5           False   76488    6221        0.075   1.032
    5            True    4164     135        0.031   0.414
    6           False     237    4408        0.949 

In [22]:
def to_odds(p): return p / (1 - p)
def to_prob(o): return o / (1 + o)

base = yA_zone.mean()
lr_group = {int(g): v for g, v in tabA["LR"].items()}
lr_if_all = {(int(r.leaf), bool(r.if_says_attack)): r.LR for r in tIF_A.itertuples()}
lr_if_used = {key: v for key, v in lr_if_all.items() if key[0] in use_if}

def agent_prob(leaf, flag, lr_if):
    out = []
    for g, f in zip(leaf, flag):
        o = to_odds(base) * lr_group[int(g)]
        key = (int(g), bool(f))
        if key in lr_if:
            o *= lr_if[key]
        out.append(float(np.clip(to_prob(o), 0.001, 0.999)))
    return np.array(out)

post = {"tree only": agent_prob(leafB, ifB, {}),
        "tree + IF (all groups)": agent_prob(leafB, ifB, lr_if_all),
        "tree + IF (rule-selected)": agent_prob(leafB, ifB, lr_if_used)}

print("combined prior, zone Brier:", round(brier_score_loss(yB_zone, pc_B[mask_B]), 4))
for name, v in post.items():
    print(f"{name:26s} zone Brier", round(brier_score_loss(yB_zone, v), 4))

combined prior, zone Brier: 0.1852
tree only                  zone Brier 0.0788
tree + IF (all groups)     zone Brier 0.0787
tree + IF (rule-selected)  zone Brier 0.0788


In [23]:
from sklearn.metrics import precision_score, recall_score, f1_score

def show(name, p):
    pred = (p > 0.5).astype(int)
    print(f"{name:28s} {brier_score_loss(y_B, p):.4f}   {precision_score(y_B, pred):.3f}   {recall_score(y_B, pred):.3f}   {f1_score(y_B, pred):.3f}")

print(f"{'method':28s} Brier    precision recall  F1")
show("raw Mahalanobis", p_B["mahalanobis"])
show("raw autoencoder", p_B["autoencoder"])
show("combined prior", pc_B)

p_agent = pc_B.copy()
p_agent[mask_B] = post["tree + IF (rule-selected)"]
show("agent", p_agent)

action = np.where(p_agent >= 0.9, "ESCALATE", np.where(p_agent >= 0.5, "HUMAN REVIEW", "LOG (low priority)"))
tri = pd.DataFrame({"action": action, "y": y_B})
print(tri.groupby("action")["y"].agg(rows="size", attack_rate="mean").round(3))
print("attacks caught by ESCALATE + REVIEW:", round(tri[tri.action != "LOG (low priority)"].y.sum() / tri.y.sum(), 3))
print("benign rows sent to ESCALATE or REVIEW:", round((tri[tri.y == 0].action != "LOG (low priority)").mean(), 3))

d = pd.DataFrame({"label": labels_B, "caught": action != "LOG (low priority)", "escalated": action == "ESCALATE"})
d = d[d.label != "BENIGN"]
out = d.groupby("label")[["caught", "escalated"]].mean().round(3)
out["rows"] = d.groupby("label").size()
print(out.sort_values("rows", ascending=False).to_string())

method                       Brier    precision recall  F1
raw Mahalanobis              0.1040   0.806   0.394   0.529
raw autoencoder              0.0977   0.906   0.424   0.578
combined prior               0.0944   0.883   0.424   0.572
agent                        0.0470   0.900   0.820   0.858
                      rows  attack_rate
action                                 
ESCALATE             25844        0.964
HUMAN REVIEW         24848        0.834
LOG (low priority)  232098        0.043
attacks caught by ESCALATE + REVIEW: 0.82
benign rows sent to ESCALATE or REVIEW: 0.022
                            caught  escalated   rows
label                                               
DoS Hulk                     0.877      0.679  23051
PortScan                     0.991      0.001  15865
DDoS                         0.631      0.631  12733
DoS GoldenEye                0.958      0.807   1046
FTP-Patator                  0.000      0.000    800
SSH-Patator                  0.002      0.

In [24]:
import json
A_DIR = "../artifacts/cicids2017"

GROUP_TEXT = {
    2: "almost no data sent forward and a very small reply header",
    3: "almost no data sent forward, with a larger reply header",
    5: "some data sent forward, with reply packet sizes that vary little",
    6: "some data sent forward, with reply packet sizes that vary a lot",
}

joblib.dump(tree, f"{A_DIR}/agent_tree.joblib")
params = {
    "base_rate": float(base),
    "lr_group": {str(g): float(v) for g, v in lr_group.items()},
    "lr_if": {f"{g},{int(flag)}": float(v) for (g, flag), v in lr_if_used.items()},
    "tree_columns": list(rawA.columns),
    "zone_low": 0.2, "zone_high": 0.9,
    "if_threshold": 0.5,
    "escalate_at": 0.9, "review_at": 0.5,
    "group_text": {str(g): t for g, t in GROUP_TEXT.items()},
}
json.dump(params, open(f"{A_DIR}/agent_params.json", "w"), indent=2)
print(sorted(os.listdir(A_DIR)))

from src.detection.agent import Agent
agent = Agent(A_DIR)
n = 60_000
sel = idx_B[:n]
scores_sel = {name: np.asarray(dets[name].predict(X[sel])["score"], dtype=float)
              for name in ("isolation_forest", "mahalanobis", "autoencoder")}
prob_sel, reasons_sel = agent.investigate(raw_test.iloc[sel], scores_sel)
print("max difference vs notebook:", float(np.abs(prob_sel - p_agent[:n]).max()))
for i in np.where(mask_B[:n])[0][:3]:
    print(round(prob_sel[i], 3), "|", reasons_sel[i])

['agent_calibrator.joblib', 'agent_params.json', 'agent_tree.joblib', 'autoencoder.pt', 'calib_autoencoder.joblib', 'calib_isolation_forest.joblib', 'calib_mahalanobis.joblib', 'iso_forest.joblib', 'mahalanobis.npz', 'preprocessor.joblib', 'thresholds.json']


/Users/sayoowo5/CS4360-Senior-Experience-Project/src/detection/detectors.py:62: RuntimeWarning: divide by zero encountered in matmul
  return ((d @ self.cov_inv) * d).sum(axis=1)
/Users/sayoowo5/CS4360-Senior-Experience-Project/src/detection/detectors.py:62: RuntimeWarning: overflow encountered in matmul
  return ((d @ self.cov_inv) * d).sum(axis=1)
/Users/sayoowo5/CS4360-Senior-Experience-Project/src/detection/detectors.py:62: RuntimeWarning: invalid value encountered in matmul
  return ((d @ self.cov_inv) * d).sum(axis=1)


max difference vs notebook: 0.0
0.838 | The detectors were unsure (24% attack), so the agent checked more evidence. | Starting point: 26% of similar uncertain records were attacks | almost no data sent forward and a very small reply header: attack more likely (odds x14.57) -> 84%
0.073 | The detectors were unsure (24% attack), so the agent checked more evidence. | Starting point: 26% of similar uncertain records were attacks | some data sent forward, with reply packet sizes that vary little: attack less likely (odds x0.22) -> 7%
0.073 | The detectors were unsure (24% attack), so the agent checked more evidence. | Starting point: 26% of similar uncertain records were attacks | some data sent forward, with reply packet sizes that vary little: attack less likely (odds x0.22) -> 7%
